# 선별 × 마스크 분리 실험 — DINOv2 + Logistic Regression (2026-10-13)

**목적:** 성능 차이가 사진 **선별**에서 오는지 **마스크**에서 오는지 나눠 본다. 빠른 모델로 모든 조합을 훑고, ConvNeXt로 확인할 조합을 고르는 단계다.

| 선별 | 내용 |
|---|---|
| 없음 | 4,000장 전부 |
| 2단계 | 2단계 선별 v3.2 (팀원 최종 판정 그대로, usable 3,062) |
| 재활용 | 기존 2단계(고정 크롭 재계산) 2,652 + 재활용 743 = 3,395 (지금까지 쓴 목록) |
| 질감 | 질감 게이트 (출력물 면적 ≥ 10%) |

| 마스크 | 내용 |
|---|---|
| 없음 | 원본 크롭 |
| 베드 지도 | 2단계 선별의 칸 판정 베드 지도 + 직선 보정 (4,000장 전부 미리 만들어 둠) |
| 질감 | 질감 마스크 (`filament_gate.mask_bed`) |
| 재활용 | 통과 사진은 베드 지도, 재활용 743장은 형상 추적 (기존 재활용 마스크 그대로) |

**조합 12개:** {없음, 2단계, 질감} × {없음, 베드 지도, 질감} 9개 + 재활용 × {질감, 재활용, 없음} 3개

**왜 이렇게:** 같은 선별에서 마스크만 바꾸면 마스크 효과, 같은 마스크에서 선별만 바꾸면 선별 효과가 보인다. 재활용 × 질감 vs 재활용 × 재활용 마스크는 사진이 같고 마스크만 다르다.

**고정한 것:** 시간 블록 5-fold + buffer 30(기존 `folds.csv`), 특징 = DINOv2 ViT-B/14 고정(CLS + 조각 평균 1,536개), 표준화(학습 fold로만) + Logistic Regression(C=1.0), 기준 0.5, 1회(결정적 계산). 학습은 각 fold 학습 구간 중 그 선별을 통과한 사진만 쓴다(장수가 조합마다 다른 것이 실험 대상). 마스크는 학습·평가에 똑같이 적용한다. **모든 조합이 4,000장 전부에 점수를 낸다**(평가 D용).

**Drive `산공종설/`에 필요한 것:** `underfill_originals_4000.zip`, `dinov2_grid_bundle_20261013.zip`(이 노트북과 같이 전달). 이전 빠른 모델 결과 폴더 `results/fast_model_v2_fullprec/`가 있으면 원본·질감 마스크 특징을 재사용해 시간이 줄어든다.

**실행:** 런타임 > 런타임 유형 변경 > GPU → 위에서부터 차례로. 오래 걸리는 건 4번(특징 추출)뿐이다. 결과는 `산공종설/results/dinov2_grid_20261013/`에 저장되고, 끊겨도 저장된 특징은 다시 계산하지 않는다.

## 1. GPU 확인, Drive 연결, 묶음 풀기

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob, sys, zipfile, shutil, json, time
DRIVE_DIR = "/content/drive/MyDrive/산공종설"      # 폴더 이름이 다르면 여기만 바꾸기
OUT = f"{DRIVE_DIR}/results/dinov2_grid_20261013"
PREV = f"{DRIVE_DIR}/results/fast_model_v2_fullprec"  # 이전 빠른 모델 결과(있으면 특징 재사용)
os.makedirs(OUT, exist_ok=True)
for f in ("underfill_originals_4000.zip", "dinov2_grid_bundle_20261013.zip"):
    assert os.path.exists(f"{DRIVE_DIR}/{f}"), f"{f} 없음: 산공종설/에 올려 주세요"
B = "/content/bundle"
if not os.path.exists(f"{B}/selections.csv"):
    with zipfile.ZipFile(f"{DRIVE_DIR}/dinov2_grid_bundle_20261013.zip") as z: z.extractall(B)
if len(glob.glob("/content/orig/**/*.jpg", recursive=True)) < 4000:
    !cp "{DRIVE_DIR}/underfill_originals_4000.zip" /content/
    !unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
sys.path.insert(0, B)
import filament_gate as fg
fg._self_check()
print(len(glob.glob("/content/orig/**/*.jpg", recursive=True)), "장 /", fg.VERSION, "/", json.load(open(f"{B}/bundle_info.json"))["counts"])

## 2. 사진 목록과 선별

이전 빠른 모델과 **같은 순서**(분할표 검증 행을 파일 이름순)로 사진을 놓는다. 그래야 저장된 특징을 그대로 재사용할 수 있다.
질감 게이트 값은 이전 결과가 있으면 가져오고, 없으면 새로 계산한다(1~2분).

In [ ]:
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor
folds = pd.read_csv(f"{B}/folds.csv", encoding="utf-8-sig")
folds["source"] = folds.source.str.replace("\\", "/", regex=False)
imgs = (folds[folds.role == "val"][["source", "label", "label_id", "frame_index", "fold"]]
        .rename(columns={"fold": "test_fold"}).sort_values("source").reset_index(drop=True))
src_idx = {s: i for i, s in enumerate(imgs.source)}
sel = pd.read_csv(f"{B}/selections.csv", encoding="utf-8-sig")
imgs = imgs.merge(sel.drop(columns=["label", "label_id", "frame_index"]), on="source", how="left", validate="1:1")
assert len(imgs) == 4000 and imgs.stage2.notna().all()

gate_file = f"{OUT}/gate_values_v2.csv"
if not os.path.exists(gate_file) and os.path.exists(f"{PREV}/gate_values_v2.csv"):
    shutil.copy(f"{PREV}/gate_values_v2.csv", gate_file); print("질감 게이트 값 재사용")
if os.path.exists(gate_file):
    gv = pd.read_csv(gate_file, encoding="utf-8-sig")
else:
    def one(s): return fg.filament_ratio(fg.crop_roi(cv2.imread(f"/content/orig/{s}")), bgr=True)
    with ThreadPoolExecutor(4) as ex: vals = list(ex.map(one, imgs.source))
    gv = pd.DataFrame({"source": imgs.source, "new": np.round(vals, 4)}); gv.to_csv(gate_file, index=False, encoding="utf-8-sig")
imgs = imgs.merge(gv[["source", "new"]].rename(columns={"new": "texture_ratio"}), on="source", how="left")
SEL = {"없음": np.ones(len(imgs), bool), "2단계": imgs.stage2.values == 1,
       "재활용": imgs.recycle.values == 1, "질감": (imgs.texture_ratio >= fg.THRESHOLD).values}
for k, m in SEL.items():
    print(f"{k:4s} 통과 {m.sum():5d}장  (정상 {m[imgs.label_id == 0].sum()}, 언더필 {m[imgs.label_id == 1].sum()})")

## 3. 마스크 적용 함수

모든 마스크는 고정 크롭(1000×1000)에서 바깥을 ImageNet 평균색 (124,116,104)으로 칠한다. 출력물 안쪽은 원본 그대로다.
- 베드 지도: 묶음의 `masks/bedmap/` (255 = 남김)
- 질감: `filament_gate.mask_bed` (게이트와 같은 지도)
- 재활용: 재활용 743장은 `masks/shape/`, 나머지는 베드 지도

In [ ]:
GRAY_BGR = (104, 116, 124)
def load_mask(rel): return cv2.imread(f"{B}/{rel}", cv2.IMREAD_GRAYSCALE)
def apply(crop_bgr, keep):
    out = crop_bgr.copy(); out[keep == 0] = GRAY_BGR; return out
def masked_crop(i, kind):
    crop = fg.crop_roi(cv2.imread(f"/content/orig/{imgs.source[i]}"))
    if kind == "nomask":  return crop
    if kind == "texture": return fg.mask_bed(crop, bgr=True)[0]
    if kind == "bedmap":  return apply(crop, load_mask(imgs.bedmap_mask[i]))
    if kind == "recycle":
        rel = imgs.shape_mask[i] if isinstance(imgs.shape_mask[i], str) and imgs.shape_mask[i] else imgs.bedmap_mask[i]
        return apply(crop, load_mask(rel))
# 자기 점검: 마스크 크기와 안쪽 보존
i0 = int(np.where(imgs.recycle_cohort == "rescue")[0][0])
for kind in ("bedmap", "recycle"):
    m = masked_crop(i0, kind); c = fg.crop_roi(cv2.imread(f"/content/orig/{imgs.source[i0]}"))
    keep = load_mask(imgs.shape_mask[i0] if kind == "recycle" else imgs.bedmap_mask[i0])
    assert m.shape == c.shape == (1000, 1000, 3) and (m[keep > 0] == c[keep > 0]).all()
print("마스크 점검 OK")

## 4. DINOv2 특징 추출 (마스크 종류별 1번씩, 저장)

- 원본(nomask)·질감(texture) 특징은 이전 빠른 모델 결과가 있으면 그대로 복사해 쓴다(같은 사진 순서·같은 코드).
- 재활용 마스크는 재활용 743장만 베드 지도와 다르므로, 베드 지도 특징을 복사한 뒤 그 743장만 새로 뽑는다.
- 처음 실행 때 DINOv2 가중치를 받는다(HF_TOKEN 경고는 무시).

In [ ]:
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as TF
IMG, MEAN, STD = 518, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
def to_input(crop_bgr):
    t = torch.from_numpy(np.ascontiguousarray(crop_bgr[:, :, ::-1])).permute(2, 0, 1)
    t = TF.resize(t, [IMG, IMG], interpolation=v2.InterpolationMode.BILINEAR, antialias=True)
    return TF.normalize(t.float() / 255.0, MEAN, STD)
dev = "cuda"
dn = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).eval().to(dev)
NPT = dn.num_prefix_tokens
@torch.no_grad()
def extract(x):
    with torch.autocast("cuda", dtype=torch.float16):
        tok = dn.forward_features(x.to(dev)).float()
    return torch.cat([tok[:, 0], tok[:, NPT:].mean(1)], 1).cpu()   # CLS + 조각 평균
class Imgs(Dataset):
    def __init__(self, idx, kind): self.idx, self.kind = list(idx), kind
    def __len__(self): return len(self.idx)
    def __getitem__(self, j): return to_input(masked_crop(self.idx[j], self.kind))
def run(idx, kind):
    out, t0 = [], time.time()
    for b, x in enumerate(DataLoader(Imgs(idx, kind), batch_size=16, num_workers=2)):
        out.append(extract(x))
        if b % 50 == 0: print(f"  {kind}: {(b + 1) * 16:5d} / {len(idx)} ({time.time() - t0:.0f}s)")
    return torch.cat(out).numpy()

X = {}
for kind, prev in (("nomask", "feats_nomask.npy"), ("texture", "feats_mask.npy"), ("bedmap", None)):
    f = f"{OUT}/feats_{kind}.npy"
    if not os.path.exists(f) and prev and os.path.exists(f"{PREV}/{prev}"):
        shutil.copy(f"{PREV}/{prev}", f); print("재사용:", kind)
    if not os.path.exists(f): np.save(f, run(range(len(imgs)), kind))
    X[kind] = np.load(f); assert X[kind].shape == (4000, 1536), (kind, X[kind].shape)
f = f"{OUT}/feats_recycle.npy"
if not os.path.exists(f):
    Xr = X["bedmap"].copy(); ridx = np.where(imgs.recycle_cohort.values == "rescue")[0]
    Xr[ridx] = run(ridx, "recycle"); np.save(f, Xr)
X["recycle"] = np.load(f)
print({k: v.shape for k, v in X.items()})

## 5. 12개 조합 학습·예측 (몇 초)

각 fold에서: 학습 = 그 fold 학습 구간(buffer 제외) ∩ 선별 통과 사진, 예측 = 그 fold 시험 구간 **전부**.
그래서 조합마다 4,000장 모두에 점수가 하나씩 생긴다(OOF). 순위는 6번의 공통 사진으로만 매긴다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
y = imgs.label_id.values
SEL_MASK = [(s, m) for s in ("없음", "2단계", "질감") for m in ("nomask", "bedmap", "texture")] + \
           [("재활용", "texture"), ("재활용", "recycle"), ("재활용", "nomask")]
MNAME = {"nomask": "없음", "bedmap": "베드 지도", "texture": "질감", "recycle": "재활용"}
def cv_prob(s, m, n_cap=None, seed=0):
    p = np.full(len(imgs), np.nan); ntr = []; rng = np.random.default_rng(seed)
    for k in range(5):
        f = folds[folds.fold == k]
        itr = f[f.role == "train"].source.map(src_idx).values; ite = f[f.role == "val"].source.map(src_idx).values
        itr = itr[SEL[s][itr]]
        if n_cap is not None and len(itr) > n_cap[k]: itr = np.sort(rng.choice(itr, n_cap[k], replace=False))
        sc = StandardScaler().fit(X[m][itr]); clf = LogisticRegression(max_iter=5000).fit(sc.transform(X[m][itr]), y[itr])
        p[ite] = clf.predict_proba(sc.transform(X[m][ite]))[:, 1]; ntr.append(len(itr))
    assert not np.isnan(p).any()
    return p, ntr
P, NTR = {}, {}
for s, m in SEL_MASK:
    P[(s, m)], NTR[(s, m)] = cv_prob(s, m)
    print(f"{s:4s} × {MNAME[m]:5s}  fold별 학습 장수 {NTR[(s, m)]}")

## 6. 평가

- **A. 주 결과 (순위는 여기서만):** 공통 사진 = 2단계 ∩ 재활용 ∩ 질감 통과 (선별 없음은 전부 포함)
- **B. 원인 분석:** A를 깨끗한 사진(옛 2단계 기준 통과) / 베드가 보이는 사진(나머지)으로 나눔
- **C. 운영:** 각 조합이 자기 선별로 판정한 사진 + 판정 비율·최장 연속 보류
- **D. 전체 4,000장:** 묶음별 — 모두 통과 / 일부만 통과 / 모두 보류. "모두 보류"에서는 정상 헛경보가 핵심(언더필 미탐은 결함이 안 보이는 사진이라 참고)

In [ ]:
PROB_T = 0.5
def metrics(p, mask):
    yy, pp = y[mask], p[mask]; pred = (pp >= PROB_T).astype(int)
    tp = int(((pred == 1) & (yy == 1)).sum()); fn = int(((pred == 0) & (yy == 1)).sum())
    fp = int(((pred == 1) & (yy == 0)).sum()); tn = int(((pred == 0) & (yy == 0)).sum())
    au = roc_auc_score(yy, pp) if 0 < yy.sum() < len(yy) else np.nan
    return dict(n=int(mask.sum()), AUROC=round(au, 4), 재현율=round(tp / max(tp + fn, 1), 4),
                오탐률=round(fp / max(fp + tn, 1), 4), 오탐=fp, 미탐=fn)
def fold_auroc(p, mask):
    return " ".join(f"{roc_auc_score(y[mask & (imgs.test_fold.values == k)], p[mask & (imgs.test_fold.values == k)]):.3f}" for k in range(5))
def longest_hold(passed):
    best = 0
    for lab in (0, 1):
        o = imgs[imgs.label_id == lab].sort_values("frame_index").index; cur = 0
        for v in passed[o]: cur = 0 if v else cur + 1; best = max(best, cur)
    return best
COMMON = SEL["2단계"] & SEL["재활용"] & SEL["질감"]
CLEAN = COMMON & (imgs.clean_old.values == 1); BEDVIS = COMMON & (imgs.clean_old.values == 0)
npass = SEL["2단계"].astype(int) + SEL["재활용"] + SEL["질감"]
ALLP, SOME, NONE = npass == 3, (npass > 0) & (npass < 3), npass == 0
print(f"공통 {COMMON.sum()} (깨끗 {CLEAN.sum()}, 베드 보임 {BEDVIS.sum()}) | 전체 묶음: 모두 통과 {ALLP.sum()}, 일부 {SOME.sum()}, 모두 보류 {NONE.sum()}")
name = lambda s, m: f"{s} × {MNAME[m]}"
A, Bt, C, D = [], [], [], []
for (s, m), p in P.items():
    A.append(dict(조합=name(s, m), **metrics(p, COMMON), fold별_AUROC=fold_auroc(p, COMMON), 학습장수=sum(NTR[(s, m)])))
    for g, msk in (("깨끗한 사진", CLEAN), ("베드 보임", BEDVIS)):
        Bt.append(dict(조합=name(s, m), 묶음=g, **metrics(p, msk)))
    C.append(dict(조합=name(s, m), 판정비율=round(SEL[s].mean(), 4), 최장연속보류=longest_hold(SEL[s]), **metrics(p, SEL[s])))
    for g, msk in (("전체 4000", np.ones(len(y), bool)), ("모두 통과", ALLP), ("일부만 통과", SOME), ("모두 보류", NONE)):
        D.append(dict(조합=name(s, m), 묶음=g, **metrics(p, msk)))
A, Bt, C, D = map(pd.DataFrame, (A, Bt, C, D))
for t, f in ((A, "A_common"), (Bt, "B_clean_vs_bed"), (C, "C_own_selection"), (D, "D_all4000_groups")):
    t.to_csv(f"{OUT}/{f}.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 250)
print("\n[A] 공통 사진 (순위)\n", A.sort_values("AUROC", ascending=False).to_string(index=False))

## 7. 한눈에 보기: 선별 × 마스크 표 (공통 사진 기준)

가로로 읽으면 **마스크 효과**, 세로로 읽으면 **선별 효과**다. 재활용 행은 마스크가 질감·재활용·없음만 있다.

In [ ]:
def grid(df, col):
    t = df.assign(선별=df.조합.str.split(" × ").str[0], 마스크=df.조합.str.split(" × ").str[1])
    return t.pivot(index="선별", columns="마스크", values=col).reindex(["없음", "2단계", "질감", "재활용"])[["없음", "베드 지도", "질감", "재활용"]]
for col in ("AUROC", "오탐", "미탐"):
    print(f"\n[A] {col}\n", grid(A, col).to_string())
for g in ("깨끗한 사진", "베드 보임"):
    print(f"\n[B] {g} — 오탐\n", grid(Bt[Bt.묶음 == g], "오탐").to_string())
print("\n[C] 운영\n", C.to_string(index=False))
print("\n[D] 모두 보류 묶음 — 정상 헛경보\n", D[D.묶음 == "모두 보류"][["조합", "n", "오탐", "오탐률"]].to_string(index=False))

## 8. 보조 확인: 학습 장수를 맞춘 버전 (선택, 몇 초)

"사진이 많아서 좋아진 것"과 "선별 덕"을 나누려고, 각 fold의 학습 장수를 **가장 적은 선별(2단계)**에 맞춰 무작위로 줄여 다시 돌린다(시드 0). 공통 사진 결과만 본다.

In [ ]:
cap = [min(NTR[(s, "nomask" if s != "재활용" else "texture")][k] for s in ("없음", "2단계", "질감", "재활용")) for k in range(5)]
E = []
for s, m in SEL_MASK:
    p, n = cv_prob(s, m, n_cap=cap, seed=0)
    E.append(dict(조합=name(s, m), **metrics(p, COMMON), 학습장수=sum(n)))
E = pd.DataFrame(E); E.to_csv(f"{OUT}/E_equal_train_size.csv", index=False, encoding="utf-8-sig")
print("fold별 학습 장수 상한:", cap); print(E.to_string(index=False))

## 9. 사진별 점수 저장 (반올림 없음)

In [ ]:
pr = imgs[["source", "label", "test_fold", "frame_index", "stage2", "recycle", "recycle_cohort", "clean_old", "texture_ratio"]].copy()
pr["질감통과"] = SEL["질감"].astype(int)
for (s, m), p in P.items(): pr[f"p_{s}_{MNAME[m]}"] = p
pr.to_csv(f"{OUT}/predictions_fullprec.csv", index=False, encoding="utf-8-sig", float_format="%.10g")
json.dump(dict(conditions=[name(s, m) for s, m in SEL_MASK], train_counts={name(*k): v for k, v in NTR.items()},
               common=int(COMMON.sum()), clean=int(CLEAN.sum()), bed_visible=int(BEDVIS.sum()),
               threshold=PROB_T, backbone="vit_base_patch14_dinov2.lvd142m", img=IMG, gate_version=fg.VERSION),
          open(f"{OUT}/run_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("저장:", sorted(os.listdir(OUT)))

## 10. 런타임 반납
결과 폴더 `results/dinov2_grid_20261013/` 전체를 받아서 공유해 주세요.

In [ ]:
from google.colab import runtime
runtime.unassign()